# Tutorial 6: SBI Surrogate Learning (`sbi_npe`)

Estimated time: 30-50 minutes

## Prerequisites — install BEFORE running this notebook

| Need | Why | Install command |
|---|---|---|
| `bayesian-metamodeling` + `[tutorials]` | Framework runtime + jupyter/matplotlib/numpy. Same as every tutorial. | `pip install -e ".[tutorials]"` |
| **`sbi` + `torch`** | **T6 only.** SBI fits the neural posterior; without these, Step 2 will skip with a preflight banner. | `pip install -e ".[sbi]"` *or* `conda install -c conda-forge pytorch sbi` |

Optional but useful: have completed Tutorial 5 (PyMC GP surrogate) so the comparison cell at the end can load both surrogate artifacts.

**If you don't want to install `sbi` right now**, that's fine: the bootstrap cell below detects this and prints a multi-line preflight banner explaining what to install, in which env, and which steps still teach. You can come back when you want to do the full SBI vs PyMC comparison.

(See Tutorial 0 for the full framework-vs-tutorial-vs-backend dependency matrix.)

## Learning aims
- Primary package aim: fit/evaluate an SBI backend with the same user-facing surrogate interface as PyMC.
- Secondary scientific aim: understand likelihood-free neural posterior estimation from simulation pairs — and the qualitative differences from PyMC's GP surrogate.

## Success criteria
- you can run SBI fit/eval and compare behavior against PyMC on matched inputs (or, if SBI is not installed, you've read the preflight banner and understand what would happen).


## Why this tutorial matters

T5 used PyMC: explicit priors, an analytical likelihood (the GP marginal over function values), MCMC under the hood. SBI takes a fundamentally different bet: it learns the posterior **directly from simulation pairs `(input, output)`**, with no likelihood you need to write down. The user-facing interface in our framework is identical (`bayesmm surrogate fit/eval`); the math underneath is completely different.

This tutorial fits SBI on the same toy data T5 used and asks the question that motivates having both backends in the first place: **do they agree?** If they do, you have two independent confirmations. If they don't, the disagreement is data — telling you something about the model that PyMC's GP smoothness assumptions and SBI's neural density estimator approximate differently.


## Step 1: Ensure training dataset exists


In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import importlib.util
import os
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)

# Preflight: detect SBI + torch. T6 invokes `bayesmm surrogate fit/eval` with
# the sbi_npe backend, which requires both `sbi` and `torch` in this kernel's
# env. Detecting their absence here lets us skip the SBI-specific steps
# cleanly with an actionable banner instead of a mid-notebook RuntimeError.
SBI_AVAILABLE = (
    importlib.util.find_spec("sbi") is not None
    and importlib.util.find_spec("torch") is not None
)

if not SBI_AVAILABLE:
    _conda_env_name = os.environ.get("CONDA_DEFAULT_ENV")
    BANNER = "=" * 72
    print()
    print(BANNER)
    print("  PREFLIGHT: SBI backend missing — Step 2 onwards will be SKIPPED")
    print(BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    if _conda_env_name:
        print(f"  Conda env name   : {_conda_env_name}")
    print("  Missing packages : 'sbi' and/or 'torch'")
    print()
    print("  Step 1 (the toy DOE run) still completes — that's the same loop")
    print("  T1 used. Step 2 (SBI fit), Step 3 (SBI eval/plot), Step 4 (PyMC")
    print("  vs SBI comparison), and the optional appendix all need SBI.")
    print()
    print("  HOW TO INSTALL — pip works regardless of conda/venv:")
    print()
    print("    # In a terminal, with THIS notebook's kernel env activated")
    if _conda_env_name:
        print(f"    # (your env: '{_conda_env_name}'):")
        print(f"    conda activate {_conda_env_name}")
    else:
        print("    # (activate it first — `conda info --envs` lists conda envs):")
    print("    pip install -e \".[sbi]\"          # via the package's extras")
    print("    # or directly:")
    print("    pip install sbi torch")
    print("    # conda alternative (sbi/torch ARE on conda-forge):")
    print("    conda install -c conda-forge pytorch sbi")
    print()
    print("  After installing, RESTART the Jupyter kernel and re-run from the top.")
    print(BANNER)


In [2]:
run_mm_cli('run', 'tutorials/specs/model.toy.grid.json')


$ bayesmm run tutorials/specs/model.toy.grid.json
Running point 1/9: {'a': 0.0, 'b': 0.0}
Running point 2/9: {'a': 0.0, 'b': 1.0}
Running point 3/9: {'a': 0.0, 'b': 2.0}
Running point 4/9: {'a': 1.0, 'b': 0.0}
Running point 5/9: {'a': 1.0, 'b': 1.0}
Running point 6/9: {'a': 1.0, 'b': 2.0}
Running point 7/9: {'a': 2.0, 'b': 0.0}
Running point 8/9: {'a': 2.0, 'b': 1.0}
Running point 9/9: {'a': 2.0, 'b': 2.0}
Stored sweep run: 14b6fd9f2f6b407a86a2f567c20f37f6
Run complete: 9 successful runs


0

## Step 2: Fit/evaluate SBI surrogate


In [ ]:
if not SBI_AVAILABLE:
    print("Step 2 SKIPPED (SBI/torch missing in this kernel) — see preflight banner above.")
else:
    run_mm_cli("surrogate", "fit", "tutorials/specs/surrogate.toy.sbi_npe.json")
    run_mm_cli(
        "surrogate", "eval", "tutorials/specs/surrogate.toy.sbi_npe.json",
        "--inputs", '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}',
        "--n", "200",
    )


## Step 3: Plot SBI predictive summary (graphic)


**Common confusion: SBI's NPE doesn't have priors in the PyMC sense.** PyMC's prior in T5 was an explicit `Normal(0, 1)` you wrote down in the model. SBI doesn't have one of those. What it has is:

- A **parameter prior** = the distribution from which simulation training inputs were drawn (here, the DOE's grid sampling defines that distribution implicitly — the grid IS the prior).
- A **learned amortized posterior** = a neural density estimator (a normalizing flow, by default `maf`) that, after training, approximates the posterior `p(θ | x)` at any query `x` without re-running the simulator.

So when you compare T5 and T6, you're comparing two completely different things: PyMC's explicit Bayesian inference with a Gaussian-process likelihood, vs SBI's neural function-fitting with the DOE as implicit prior. They both give you `(mean, std)` per query point — but the meanings of those numbers come from different machinery.


In [ ]:
if not SBI_AVAILABLE:
    print("Step 3 SKIPPED (SBI/torch missing) — see preflight banner above.")
    sbi_mean = None
    sbi_std = None
    sbi_inputs = None
else:
    import json
    import numpy as np
    import matplotlib.pyplot as plt
    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate
    # `root` came from bootstrap() at the top of the notebook — no need for path autodetect here.
    spec_payload = json.loads((root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text())
    spec = SurrogateSpec.model_validate(spec_payload)
    inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)
    mean = np.asarray(result["summary"]["mean"], dtype=float)
    std = np.asarray(result["summary"].get("std", [0.0] * len(mean)), dtype=float)
    x = np.arange(len(mean))
    plt.figure(figsize=(6.5, 4))
    plt.errorbar(x, mean, yerr=std, fmt="o-", capsize=4, color="tab:orange")
    plt.title("SBI surrogate predictive mean ± std")
    plt.xlabel("query point index")
    plt.ylabel("predicted y")
    plt.grid(True, alpha=0.3)
    plt.show()
    # Stash predictions for the side-by-side comparison cell below.
    sbi_mean = mean
    sbi_std = std
    sbi_inputs = inputs
    print(f"SBI predictive mean:  {[float(f'{m:.4f}') for m in sbi_mean]}")
    print(f"SBI predictive std :  {[float(f'{s:.4f}') for s in sbi_std]}")

## Step 4: Compare PyMC and SBI on the same query points

This is the lesson — the rest of T6 was setup. We load T5's PyMC surrogate (already fitted in T5) and evaluate it on the same 4 query points, then plot the two predictions on shared axes.


In [ ]:
if not SBI_AVAILABLE or sbi_mean is None:
    print("Step 4 SKIPPED — needs the SBI predictions from Step 3.")
    print("If you want to see the comparison, install SBI (preflight banner above)")
    print("and re-run from the top.")
else:
    import numpy as np
    import matplotlib.pyplot as plt
    import json
    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate
    # Load and evaluate T5's PyMC surrogate on the same 4 query points used above.
    pymc_spec_payload = json.loads((root / "tutorials/specs/surrogate.toy.pymc_gp.json").read_text())
    pymc_spec = SurrogateSpec.model_validate(pymc_spec_payload)
    pymc_result = eval_surrogate(spec=pymc_spec, inputs_payload=sbi_inputs, n=300)
    pymc_mean = np.asarray(pymc_result["summary"]["mean"], dtype=float)
    pymc_std = np.asarray(pymc_result["summary"].get("std", [0.0] * len(pymc_mean)), dtype=float)
    # Plot the two predictive series on shared axes, with `a + b` (the analytical truth) as x.
    query_x = np.asarray(sbi_inputs["a"]) + np.asarray(sbi_inputs["b"])
    truth_x = np.linspace(0, 4, 50)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(truth_x, truth_x, "k:", alpha=0.5, label="analytical truth: y = a + b")
    ax.errorbar(
        query_x - 0.04, pymc_mean, yerr=pymc_std, fmt="o", color="tab:blue",
        markersize=10, capsize=5, label=f"PyMC GP (mean ± std={pymc_std.mean():.1e})",
    )
    ax.errorbar(
        query_x + 0.04, sbi_mean, yerr=sbi_std, fmt="s", color="tab:orange",
        markersize=10, capsize=5, label=f"SBI NPE (mean ± std={sbi_std.mean():.2f})",
    )
    ax.set_title("PyMC vs SBI on the same query points (toy: y = a + b)")
    ax.set_xlabel("a + b")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"\nNumeric comparison (mean predictions, then std):")
    print(f"  truth (a+b)  : {[float(f'{t:.4f}') for t in query_x]}")
    print(f"  PyMC mean    : {[float(f'{m:.4f}') for m in pymc_mean]}")
    print(f"  SBI mean     : {[float(f'{m:.4f}') for m in sbi_mean]}")
    print(f"  PyMC std     : {[float(f'{s:.2e}') for s in pymc_std]}  (essentially 0 — GP is sure)")
    print(f"  SBI std      : {[float(f'{s:.4f}') for s in sbi_std]}  (a real width — neural posterior)")

## Mini-lesson: how to read the comparison

The two surrogates give you `(mean, std)` per query point — but the std values come from very different machinery:

- **PyMC GP std** is essentially zero on this toy (~1e-6). The GP's prior over function shape was good (smooth kernel), the data was perfectly fit by a 9-point grid, and the marginal posterior collapses to a near-delta.
- **SBI NPE std** is real (~0.1-0.5). Even on data where the truth is exactly recoverable, the neural density estimator carries finite training-noise width. That's the cost of *not* using a likelihood.

**The lesson is what to do when they disagree.** On THIS toy they barely disagree on the mean (both land near `a + b`). But on a realistic biological model, you might see PyMC say `0.4 ± 0.05` and SBI say `0.6 ± 0.3`. That disagreement is data:

- If PyMC's tight band is *inside* SBI's wide band → SBI is just less confident; defer to PyMC where you trust its kernel choice.
- If the two means are *well-separated* (say >2 std apart) → one of the surrogates' assumptions is wrong for this region. Don't trust either prediction without re-running the simulator at that input.
- If both are confident but disagree → the kernel and the neural density estimator are both extrapolating, and they're extrapolating differently. This usually means there's no nearby training data — the surrogates are guessing.

**SBI shines when likelihoods are hard to write** (multi-modal models, simulators with discrete branches, models without a closed-form pdf). PyMC shines when the GP's smoothness prior is appropriate (most physical models, especially when you want extrapolation guarantees). Have both available; cross-check on every coupled-model task.


## Optional appendix: confidence check that the SBI backend works

The cell below runs a single regression test from the package's own test suite. It's not part of the lesson — just an "is my SBI install actually functional" smoke test you can run if anything earlier surprised you. Skip on first read.


In [ ]:
if not SBI_AVAILABLE:
    print("Optional appendix SKIPPED (SBI missing in this kernel).")
else:
    run_tool(
        "pytest", "-q", "tests/test_surrogate_backends.py",
        "-k", "sbi_npe_backend_fit_sample_and_logprob",
        check=False,
    )


## Predict how SBI will differ

You now have two surrogates trained on identical data: T5's GP and this notebook's
neural posterior estimator. Before comparing them, predict:

1. **Between** training points, will they largely agree or clearly disagree?
2. **Outside** the training range, which one reverts to a prior-like mean, and which
   is more likely to produce something confidently arbitrary?
3. Which needs **more data** before its uncertainty means anything?

*(Expected: close agreement in the interior — both are fitting the same smooth
function. Outside, the GP degrades gracefully toward its prior mean with a widening
band, which is a well-understood failure mode; the neural estimator has no such
guarantee and can be confidently wrong off-distribution. SBI is the hungrier of the
two: 9 points is very little for a density estimator.)*

**Why this comparison is the lesson:** picking a surrogate backend is not a
performance question, it's a question of *which failure mode you can live with*.

## Recap: what T6 established

- **Backends are swappable behind one spec contract.** Changing `backend` from
  `pymc_gp` to `sbi_npe` re-points the whole pipeline; nothing else in the spec moves.
- A **neural posterior estimator** learns a density directly rather than assuming a
  smooth prior over functions — more flexible, hungrier for data, and without the
  GP's graceful off-distribution degradation.
- Two surrogates on identical data can **agree in the interior and diverge outside
  it**. Which you choose is a decision about acceptable failure modes, not accuracy.

One sentence to carry forward: *the spec contract is the abstraction — backends are
an implementation detail you can revisit without rewriting your pipeline.*

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: SBI backend missing` | `sbi` and/or `torch` absent from this kernel | `conda env create -f environment-sbi.yml`, then use the `py312_bayesmm_sbi` kernel. |
| `Maximum number of epochs reached, but network has not yet fully converged` | Training hit its epoch cap | Expected on tiny data. Note the framework pins `sbi<0.27` — on 0.27 this warning appears routinely and, under `filterwarnings = error`, becomes a hard test failure. |
| Results differ between runs | Neural training is stochastic | Set the seed in the surrogate spec. Unlike the GP, SBI will not reproduce bit-for-bit without one. |
| Predictions are wild outside the training range | Density estimators do not extrapolate | Not a bug — see the prediction exercise above. Use the GP if out-of-range behaviour matters. |
| Import succeeds but fitting errors on a kwarg | sbi changed its API between versions | The backend carries two compat layers (`tracker=` vs legacy `summary_writer=`). Versions outside 0.22–0.26 are untested. |

## Final check: T6's SBI surrogate ran (or skipped cleanly)

Two-tier assertion:
- if `SBI_AVAILABLE`: latest SBI artifact for the T6 spec exists with non-empty `posterior_draws`; mean predictions are within absolute error 0.5 of `y = a + b` (SBI is noisier than PyMC GP, hence the looser tolerance);
- else: the preflight banner was printed and Step 2+ was deliberately skipped.

In [ ]:
# Self-check: SBI surrogate produced (or preflight skipped cleanly).
import json as _json
import numpy as _np
if not SBI_AVAILABLE:
    print(f"\n[T6 self-check OK] Step 2+ skipped per preflight (SBI_AVAILABLE={SBI_AVAILABLE}).")
else:
    from pathlib import Path as _P
    _arts = sorted((root / "tmp/surrogate_artifacts").glob("*/artifact.json"),
                   key=lambda p: p.stat().st_mtime)
    _sbi_arts = []
    for _a in _arts:
        _p = _json.loads(_a.read_text())
        if _p.get("backend") == "sbi_npe":
            _sbi_arts.append(_a)
    assert _sbi_arts, "No SBI surrogate artifact produced — Step 2's fit didn't run."
    _latest = _sbi_arts[-1]
    from bayesian_metamodeling.spec import SurrogateSpec
    from bayesian_metamodeling.surrogates import eval_surrogate
    _spec = SurrogateSpec.model_validate(_json.loads((root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text()))
    _inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
    _result = eval_surrogate(spec=_spec, inputs_payload=_inputs, n=200)
    _mean = _np.asarray(_result["summary"]["mean"], dtype=float)
    _truth = _np.asarray(_inputs["a"]) + _np.asarray(_inputs["b"])
    _mae = float(_np.mean(_np.abs(_mean - _truth)))
    # SBI is noisier than the PyMC GP — looser tolerance.
    assert _mae < 0.5, f"SBI mean prediction MAE={_mae:.4f} > 0.5 — surrogate not converged."
    print(f"\n[T6 self-check OK] SBI NPE MAE={_mae:.5f} (< 0.5); artifact at {_latest.relative_to(root)}")
